In [1]:
import pickle

import pandas as pd
import numpy as np
import os
from config_paths import DATA_FOLDER, UKHLS_WAVES
from config_variables import VARIABLE_MAP

# --- PATH CONFIGURATION ---
WAVES = list(UKHLS_WAVES)
RAW_DIR = f"../{DATA_FOLDER}/0_raw/ukhls"
PICKLE_DIR = f"../{DATA_FOLDER}/2_pickle_ukhls_waves"
SIPHER_PIDP_PKL = f"../{DATA_FOLDER}/1_pickle_sipher/sipher_unique_pidp.pkl"


def load_sipher_pidp_set(pkl_path: str) -> frozenset:
    """Distinct pidp values from step 1 (1_pickle_sipher writes sipher_unique_pidp.pkl)."""
    if not os.path.exists(pkl_path):
        raise FileNotFoundError(
            f"Sipher unique-pidp pickle not found: {pkl_path}\n"
            "Run 1_pickle_sipher.ipynb first (writes sipher_unique_pidp.pkl next to sipher_optimized.pkl)."
        )
    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)
    if not isinstance(obj, frozenset):
        raise TypeError(f"Expected frozenset in {pkl_path}, got {type(obj)}")
    return obj


def _filter_to_sipher_pidp(df: pd.DataFrame, sipher_pidp: frozenset, label: str) -> pd.DataFrame:
    pidp_key = pd.to_numeric(df["pidp"], errors="coerce")
    keep = pidp_key.notna() & pidp_key.astype("int64").isin(sipher_pidp)
    n_before = len(df)
    out = df.loc[keep].copy()
    out["pidp"] = pidp_key.loc[keep].astype("int64")
    n_drop = n_before - len(out)
    if n_drop:
        print(f"   -> {label}: dropped {n_drop:,} rows not in Sipher ({n_before:,} -> {len(out):,}).")
    return out


def run_standardized_ingestion(sipher_pidp: frozenset) -> None:
    if not os.path.exists(PICKLE_DIR):
        os.makedirs(PICKLE_DIR)

    for w in WAVES:
        ind_path = os.path.join(RAW_DIR, f"{w}_indresp.tab")
        hh_path = os.path.join(RAW_DIR, f"{w}_hhresp.tab")

        if not os.path.exists(ind_path):
            print(f"Skipping Wave {w}: Individual response file not found.")
            continue

        print(f"--- Ingesting Wave {w} ---")

        # 1. SCAN HEADERS
        ind_headers = pd.read_csv(ind_path, sep='\t', nrows=0).columns.tolist()
        hh_exists = os.path.exists(hh_path)
        hh_headers = pd.read_csv(hh_path, sep='\t', nrows=0).columns.tolist() if hh_exists else []

        # 2. SELECT RELEVANT COLUMNS BASED ON VARIABLE_MAP
        ind_to_load = ['pidp']
        hidp_col_name = next((c for c in [f"{w}_hidp", 'hidp'] if c in ind_headers), None)
        if hidp_col_name:
            ind_to_load.append(hidp_col_name)

        hh_to_load = []
        if hh_exists:
            hh_hidp = next((c for c in [f"{w}_hidp", 'hidp'] if c in hh_headers), None)
            if hh_hidp:
                hh_to_load.append(hh_hidp)

        for base in VARIABLE_MAP.keys():
            if base == 'pidp':
                continue
            prefixed = f"{w}_{base}"
            if prefixed in ind_headers:
                ind_to_load.append(prefixed)
            elif prefixed in hh_headers:
                hh_to_load.append(prefixed)

        # 3. LOAD INDIVIDUAL AND HOUSEHOLD DATA
        df_ind = pd.read_csv(ind_path, sep='\t', usecols=ind_to_load, low_memory=False)

        if hh_to_load and len(hh_to_load) > 1:
            df_hh = pd.read_csv(hh_path, sep='\t', usecols=hh_to_load, low_memory=False)
            df = pd.merge(df_ind, df_hh, on=hidp_col_name, how='left')
            print(f"   -> Merged {len(hh_to_load) - 1} household variables.")
        else:
            df = df_ind

        df = _filter_to_sipher_pidp(df, sipher_pidp, f"Wave {w}")

        # 4. SAVE PICKLE
        out_file = os.path.join(PICKLE_DIR, f"{w}_indresp_optimized.pkl")
        df.to_pickle(out_file, protocol=5)
        print(f"   -> Saved {len(df.columns)} variables for {len(df):,} rows to {out_file}\n")


def pickle_xwavedat(sipher_pidp: frozenset) -> None:
    xwave_path = os.path.join(RAW_DIR, "xwavedat.tab")
    if not os.path.exists(xwave_path):
        print("xwavedat.tab not found — skipping.")
        return
    print("--- Ingesting xwavedat ---")
    df = pd.read_csv(xwave_path, sep='\t', low_memory=False)
    if "pidp" not in df.columns:
        raise KeyError("xwavedat.tab has no pidp column")
    df = _filter_to_sipher_pidp(df, sipher_pidp, "xwavedat")
    out_file = os.path.join(PICKLE_DIR, "xwavedat.pkl")
    df.to_pickle(out_file, protocol=5)
    size_mb = os.path.getsize(out_file) / (1024 * 1024)
    print(f"   -> Saved {len(df.columns)} variables for {len(df):,} rows to {out_file}")
    print(f"   -> File size: {size_mb:.1f} MB\n")


if __name__ == "__main__":
    print(f"Loading Sipher unique pidp from {SIPHER_PIDP_PKL} ...")
    _sipher_pidp = load_sipher_pidp_set(SIPHER_PIDP_PKL)
    print(f"   -> {len(_sipher_pidp):,} unique pidp\n")
    run_standardized_ingestion(_sipher_pidp)
    pickle_xwavedat(_sipher_pidp)



🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  

Loading Sipher unique pidp from ../data/1_pickle_sipher/sipher_unique_pidp.pkl ...
   -> 27,330 unique pidp

--- Ingesting Wave o ---
   -> Wave o: dropped 13,231 rows not in Sipher (32,849 -> 19,618).
   -> Saved 35 variables for 19,618 rows to ../data/2_pickle_ukhls_waves/o_indresp_optimized.pkl

--- Ingesting Wave n ---
   -> Wave n: dropped 14,742 rows not in Sipher (35,471 -> 20,729).
   -> Saved 36 variables for 20,729 rows to ../data/2_pickle_ukhls_waves/n_indresp_optimized.pkl

--- Ingesting Wave m ---
   -> Wave m: dropped 6,419 rows not in Sipher (27,998 -> 21,579).
   -> Saved 32 variables for 21,579 rows to ../data/2_pickle_ukhls_waves/m_indresp_optimized.pkl

--- Ingesting